# Keeping Customers Before They Leave: A Data-Driven Retention Proposal for Company A
**GCI World 2026 September — Final Assignment (Course Project PoC)**  
**Author:** Islam Raad (`Islam-Raad`) — AI Engineering, University of Mosul

---

## 0. Overview & How to Run

### Purpose
This self-contained notebook implements the end-to-end analysis supporting our Proof-of-Concept (PoC) business proposal for **Company A**, an anonymous wireless telecommunications operator (~100,000 customers):
1. **Data Audit & Exploratory Data Analysis (EDA):** Structural checks, missingness patterns, documentation vs. data verification, and statistical robustness testing of lifecycle churn thresholds (~300-day handset age and months 10–12 tenure cliff).
2. **Leakage-Safe Preprocessing & Feature Engineering:** Domain-grounded behavioral, lifecycle, call-quality, and economic features.
3. **Predictive Modeling & Interpretation:** Stratified 5-fold cross-validation across baseline and boosted-tree models, held-out test evaluation, top-$k$ targeting lift, gain and permutation importance, partial dependence, and fairness checks.
4. **Realistic Business-Impact Model:** Label-shift re-basing from the 49.56% resampled dataset to realistic telecom churn priors (5%–10%), net value estimation in monetary units (MU), scenario analysis, break-even success rate ($s^*$), optimal targeting depth ($k^*$), and tornado sensitivity.

### Folder Structure & Auto-Detection
- The notebook automatically locates `Client.csv` and `Record.csv` by checking `./telecom/`, `../telecom/` (when run from `submission/`), and `/content/drive/MyDrive/Final Assignment/telecom/` (Google Colab).
- When run inside the project workspace, it writes summary metrics to `outputs/results.json`, tables to `outputs/tables/`, and 200-DPI `matplotlib` figures to `outputs/figures/`.

### Generative AI Disclosure
Generative AI tools (Claude and Google Antigravity / Gemini) were used as analytical and coding assistants during project development. All code was executed and verified against the raw dataset, all external market statistics were verified from primary sources, and the full methodology is documented and explained by the author.

> **So what:** Every figure and number in the 15-slide executive presentation is reproducibly generated from this single notebook with `random_state=42`.

---
## 1. Setup, Environment & Matplotlib Styling Helpers

In [1]:
import json
import os
import platform
import sys
import time
import warnings
from pathlib import Path

import joblib
import matplotlib
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import sklearn
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from xgboost import XGBClassifier

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False

warnings.filterwarnings('ignore', category=FutureWarning)
SEED = 42
np.random.seed(SEED)
pd.set_option('display.max_columns', 120)

# Auto-detect DATA_DIR and OUTPUT_DIR (works from repo root, submission/, or Colab)
CANDIDATE_DATA_DIRS = [
    Path('./telecom'),
    Path('../telecom'),
    Path('/content/drive/MyDrive/Final Assignment/telecom'),
]
DATA_DIR = None
for cand in CANDIDATE_DATA_DIRS:
    if (cand / 'Client.csv').exists() and (cand / 'Record.csv').exists():
        DATA_DIR = cand.resolve()
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        'Could not locate telecom dataset. Expected Client.csv and Record.csv in one of: '
        + ', '.join(str(p) for p in CANDIDATE_DATA_DIRS)
    )

PROJECT_ROOT = DATA_DIR.parent
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
FIG_DIR = OUTPUT_DIR / 'figures'
TAB_DIR = OUTPUT_DIR / 'tables'
MOD_DIR = OUTPUT_DIR / 'models'
for d in (OUTPUT_DIR, FIG_DIR, TAB_DIR, MOD_DIR):
    d.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = OUTPUT_DIR / 'results.json'
if RESULTS_PATH.exists():
    RESULTS = json.loads(RESULTS_PATH.read_text(encoding='utf-8'))
else:
    RESULTS = {
        'data': {},
        'eda': {},
        'models': [],
        'final_model': {},
        'topk': [],
        'importance': {},
        'impact': {},
        'meta': {},
    }

# Font check & Matplotlib rcParams (PRD §10 — matplotlib only)
available_fonts = {f.name for f in fm.fontManager.ttflist}
ACTIVE_FONT = 'Open Sans' if 'Open Sans' in available_fonts else 'DejaVu Sans'

PALETTE = {
    'navy': '#0B1F3A',
    'churn': '#E4572E',     # Coral (risk / churn)
    'stayed': '#3B7DD8',    # Blue (stayed / baseline)
    'teal': '#0F8B8D',      # Teal (highlight / accent)
    'amber': '#F2A541',     # Amber (callout)
    'text': '#1F2933',      # Neutral dark text
    'slate': '#5B6B7B',     # Secondary text / arrows
    'grid': '#D9E0E7',      # Light horizontal grid
    'card': '#F5F7FA',      # Card / panel background
    'white': '#FFFFFF',
}

FIG_SIZES = {
    'full': (9.0, 3.6),
    'half': (4.4, 3.3),
    'wide_short': (9.0, 2.4),
    'square': (4.4, 4.0),
}

TEAL_CMAP = LinearSegmentedColormap.from_list(
    'white_teal', ['#FFFFFF', '#D5ECEC', '#6BBEC0', '#0F8B8D', '#084C4E']
)

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Open Sans', 'DejaVu Sans'],
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.labelcolor': PALETTE['text'],
    'axes.edgecolor': PALETTE['grid'],
    'axes.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'xtick.color': PALETTE['slate'],
    'ytick.color': PALETTE['slate'],
    'legend.fontsize': 10,
    'legend.frameon': False,
    'lines.linewidth': 2.2,
    'lines.markersize': 5.5,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.dpi': 200,
    'savefig.facecolor': 'white',
})


def make_fig(size_key='full', nrows=1, ncols=1, **kwargs):
    """Create a matplotlib figure at exact slide placement size with constrained_layout."""
    figsize = FIG_SIZES.get(size_key, size_key)
    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols, figsize=figsize, constrained_layout=True, **kwargs
    )
    return fig, axes


def style_ax(ax, grid_axis='y', caption=None):
    """Apply clean executive styling (no top/right spines, subtle grid, optional n= caption)."""
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color(PALETTE['slate'])
    ax.spines['bottom'].set_color(PALETTE['slate'])
    if grid_axis in ('y', 'both'):
        ax.yaxis.grid(True, color=PALETTE['grid'], linewidth=0.8, linestyle='-')
        ax.set_axisbelow(True)
    if grid_axis in ('x', 'both'):
        ax.xaxis.grid(True, color=PALETTE['grid'], linewidth=0.8, linestyle='-')
        ax.set_axisbelow(True)
    if caption:
        ax.text(
            0.99, -0.18, caption, transform=ax.transAxes,
            ha='right', va='top', fontsize=9.5, color=PALETTE['slate']
        )
    return ax


def save_slide_fig(fig, filename):
    """Save figure at 200 DPI without bbox_inches='tight' so placed dimensions stay exact."""
    out_path = FIG_DIR / filename
    fig.savefig(out_path, dpi=200, facecolor='white')
    return out_path


def save_results():
    """Persist RESULTS dictionary to outputs/results.json."""
    RESULTS_PATH.write_text(json.dumps(RESULTS, indent=2), encoding='utf-8')


RESULTS['meta'] = {
    'seed': SEED,
    'interpreter': sys.executable,
    'python_version': platform.python_version(),
    'active_font': ACTIVE_FONT,
    'versions': {
        'pandas': pd.__version__,
        'numpy': np.__version__,
        'scikit-learn': sklearn.__version__,
        'xgboost': xgb.__version__,
        'matplotlib': matplotlib.__version__,
        'scipy': scipy.__version__,
        'joblib': joblib.__version__,
        'lightgbm': lgb.__version__ if HAS_LIGHTGBM else None,
    },
}
save_results()
print(f'DATA_DIR: {DATA_DIR}')
print(f'Active font: {ACTIVE_FONT}')
print('Versions:', json.dumps(RESULTS['meta']['versions'], indent=2))

DATA_DIR: /home/islam-raad/Projects/Final Assignment/telecom
Active font: DejaVu Sans
Versions: {
  "pandas": "3.0.2",
  "numpy": "2.4.4",
  "scikit-learn": "1.8.0",
  "xgboost": "3.4.1",
  "matplotlib": "3.10.8",
  "scipy": "1.17.1",
  "joblib": "1.5.3",
  "lightgbm": "4.7.0"
}


> **So what:** Our environment, random seed (`42`), data paths, and executive `matplotlib` design system are locked in, ensuring every chart and metric is reproducible.

---
## 2. Load & Merge Dataset

Company A's dataset is provided as two customer-level tables in `telecom/`:
- `Client.csv`: 100,000 rows × 50 columns (`Customer_ID` + 49 demographic, household, handset, and lifetime summary features; 21 `object` and 28 numeric columns).
- `Record.csv`: 100,000 rows × 51 columns (`Customer_ID` + 49 numeric monthly usage, billing, call-quality, and customer-care features + the binary target `churn`). Note that `months` (service tenure) resides in `Record.csv`.

In [2]:
client_path = DATA_DIR / 'Client.csv'
record_path = DATA_DIR / 'Record.csv'

client = pd.read_csv(client_path)
record = pd.read_csv(record_path)

print(f'Client shape: {client.shape}')
print(f'Record shape: {record.shape}')

# Strict schema & key assertions (T0.5)
assert client.shape == (100000, 50), f'Unexpected Client shape: {client.shape}'
assert record.shape == (100000, 51), f'Unexpected Record shape: {record.shape}'
assert 'Customer_ID' in client.columns and 'Customer_ID' in record.columns
assert client['Customer_ID'].is_unique, 'Client Customer_ID is not unique'
assert record['Customer_ID'].is_unique, 'Record Customer_ID is not unique'
assert set(client['Customer_ID']) == set(record['Customer_ID']), 'Customer_ID sets do not match'
assert (client['Customer_ID'].values == record['Customer_ID'].values).all(), 'Customer_ID ordering differs'

# 1-to-1 inner merge on Customer_ID
df = record.merge(client, on='Customer_ID', how='inner', validate='one_to_one')
assert df.shape == (100000, 100), f'Unexpected merged shape: {df.shape}'

print(f'Merged shape: {df.shape}')
print(f'Customer_ID range: {int(df["Customer_ID"].min())} to {int(df["Customer_ID"].max())} (unique: {df["Customer_ID"].nunique()})')
print(f'Tenure (months) in Record.csv: min={int(df["months"].min())}, median={float(df["months"].median()):.1f}, max={int(df["months"].max())}')
print('\nAll 100 columns in merged DataFrame (df.columns):')
print(list(df.columns))

RESULTS['data'].update({
    'n_rows': int(df.shape[0]),
    'n_cols': int(df.shape[1]),
    'n_features_raw': int(df.shape[1] - 2),
    'client_shape': list(client.shape),
    'record_shape': list(record.shape),
    'id_min': int(df['Customer_ID'].min()),
    'id_max': int(df['Customer_ID'].max()),
    'months_min': int(df['months'].min()),
    'months_median': float(df['months'].median()),
    'months_max': int(df['months'].max()),
})
save_results()

Client shape: (100000, 50)
Record shape: (100000, 51)
Merged shape: (100000, 100)
Customer_ID range: 1000001 to 1100000 (unique: 100000)
Tenure (months) in Record.csv: min=6, median=16.0, max=61

All 100 columns in merged DataFrame (df.columns):
['rev_Mean', 'mou_Mean', 'totmrc_Mean', 'da_Mean', 'ovrmou_Mean', 'ovrrev_Mean', 'vceovr_Mean', 'datovr_Mean', 'roam_Mean', 'change_mou', 'change_rev', 'drop_vce_Mean', 'drop_dat_Mean', 'blck_vce_Mean', 'blck_dat_Mean', 'unan_vce_Mean', 'unan_dat_Mean', 'plcd_vce_Mean', 'plcd_dat_Mean', 'recv_vce_Mean', 'recv_sms_Mean', 'comp_vce_Mean', 'comp_dat_Mean', 'custcare_Mean', 'ccrndmou_Mean', 'cc_mou_Mean', 'inonemin_Mean', 'threeway_Mean', 'mou_cvce_Mean', 'mou_cdat_Mean', 'mou_rvce_Mean', 'owylis_vce_Mean', 'mouowylisv_Mean', 'iwylis_vce_Mean', 'mouiwylisv_Mean', 'peak_vce_Mean', 'peak_dat_Mean', 'mou_peav_Mean', 'mou_pead_Mean', 'opk_vce_Mean', 'opk_dat_Mean', 'mou_opkv_Mean', 'mou_opkd_Mean', 'drop_blk_Mean', 'attempt_Mean', 'complete_Mean', 'cal

> **So what:** `Client.csv` (100,000 × 50) and `Record.csv` (100,000 × 51) join 1-to-1 on `Customer_ID` (`1000001`–`1100000`) with zero orphan or duplicate records, producing a 100,000 × 100 master table (`98` raw features + `Customer_ID` + `churn`) where every customer has at least 6 months of tenure.

---
## 3. Data Audit & Quality Assessment

### 3.1 Structure, Data Types, Memory & Duplicate Checks (T1.1)
We audit the merged 100,000 × 100 table (`df`) for column data types (`float64`, `int64`, `object`/`str`), memory footprint, full-row duplicates, feature-only duplicates (excluding `Customer_ID`), and `Customer_ID` continuity.

In [3]:
# Note: pandas 3.0+ may represent text columns as 'str' or 'object'; count both text vs numeric types cleanly
raw_dtypes = df.dtypes.astype(str).value_counts().to_dict()
n_float64 = int((df.dtypes == 'float64').sum())
n_int64 = int((df.dtypes == 'int64').sum())
n_text = int(df.shape[1] - n_float64 - n_int64)
text_cols = [c for c in df.columns if df[c].dtype not in ('float64', 'int64')]
float_cols = [c for c in df.columns if df[c].dtype == 'float64']
int_cols = [c for c in df.columns if df[c].dtype == 'int64']

mem_mb = float(df.memory_usage(deep=True).sum() / (1024 ** 2))
n_dup_rows = int(df.duplicated().sum())
n_dup_without_id = int(df.drop(columns=['Customer_ID']).duplicated().sum())
id_expected = np.arange(1000001, 1100001)
id_exact_seq = bool(np.array_equal(df['Customer_ID'].values, id_expected))

assert n_float64 == 69, f'Expected 69 float64 cols, got {n_float64}'
assert n_int64 == 10, f'Expected 10 int64 cols, got {n_int64}'
assert n_text == 21, f'Expected 21 text/object cols, got {n_text}'
assert n_dup_rows == 0 and n_dup_without_id == 0
assert id_exact_seq

print(f'Dtype summary: float64={n_float64}, int64={n_int64}, object/string={n_text} (raw pandas dtypes: {raw_dtypes})')
print(f'Deep memory footprint: {mem_mb:.2f} MB')
print(f'Duplicate full rows: {n_dup_rows} | Duplicate feature rows (excl. Customer_ID): {n_dup_without_id}')
print(f'Customer_ID exact sequential 1000001..1100000: {id_exact_seq}')
print(f'int64 columns ({len(int_cols)}): {int_cols}')
print(f'Text/categorical columns ({len(text_cols)}): {text_cols}')

RESULTS['data']['dtypes'] = {
    'float64': n_float64,
    'int64': n_int64,
    'object': n_text,
    'int64_cols': int_cols,
    'object_cols': text_cols,
}
RESULTS['data']['memory_mb'] = round(mem_mb, 2)
RESULTS['data']['duplicate_rows'] = n_dup_rows
RESULTS['data']['duplicate_feature_rows'] = n_dup_without_id
RESULTS['data']['id_sequential_1000001_1100000'] = id_exact_seq
save_results()

Dtype summary: float64=69, int64=10, object/string=21 (raw pandas dtypes: {'float64': 69, 'str': 21, 'int64': 10})
Deep memory footprint: 158.88 MB
Duplicate full rows: 0 | Duplicate feature rows (excl. Customer_ID): 0
Customer_ID exact sequential 1000001..1100000: True
int64 columns (10): ['churn', 'months', 'Customer_ID', 'uniqsubs', 'actvsubs', 'totcalls', 'adjqty', 'avg3mou', 'avg3qty', 'avg3rev']
Text/categorical columns (21): ['new_cell', 'crclscod', 'asl_flag', 'prizm_social_one', 'area', 'dualband', 'refurb_new', 'hnd_webcap', 'ownrent', 'dwlltype', 'marital', 'infobase', 'HHstatin', 'dwllsize', 'ethnic', 'kid0_2', 'kid3_5', 'kid6_10', 'kid11_15', 'kid16_17', 'creditcd']


> **So what:** The merged dataset has `69` `float64`, `10` `int64` (including `Customer_ID`, `churn`, `months`, household subscriber and handset counts), and `21` categorical text columns (all from `Client.csv`), with zero duplicate records and sequential IDs `1,000,001`–`1,100,000`.

---
## 4. Exploratory Data Analysis & Robustness Checks
*Pending Phase 2 (T2.1–T2.11) — Device age (~300d) robustness, tenure cliff (months 10–12), entanglement, usage collapse, revenue, care calls, and fairness.*

---
## 5. Problem Definition & Decision Framing
*Pending Phase 4 (T4.1–T4.4) — Decision matrix, problem statement, target/task justification, and operational KPI.*

---
## 6. Leakage-Safe Preprocessing & Feature Engineering
*Pending Phase 5 (T5.1–T5.4) — Pipelines, domain features, leakage audit, and fairness variants.*

---
## 7. Predictive Modeling & Experiments

### 7.1 Tutorial Baseline Calibration Check (T0.6)
Before building our leakage-safe pipelines and domain-engineered features (Phases 5–6), we reproduce the exact `tutorial.ipynb` baseline (`LabelEncoder` on string-cast `object` columns + `XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, eval_metric='logloss', random_state=42)` on a 70/30 stratified split with `random_state=42`) to verify end-to-end calibration.

In [4]:
# Exact tutorial preprocessing & split
df_tut = df.drop(columns=['Customer_ID']).copy()
object_cols_tut = df_tut.select_dtypes(include='object').columns.tolist()

for col in object_cols_tut:
    le = LabelEncoder()
    df_tut[col] = le.fit_transform(df_tut[col].astype(str))

X_tut = df_tut.drop('churn', axis=1)
y_tut = df_tut['churn']

X_train_tut, X_test_tut, y_train_tut, y_test_tut = train_test_split(
    X_tut, y_tut, test_size=0.3, random_state=SEED, stratify=y_tut
)

# Majority-class baseline (M0) on test split
maj_class = int(y_train_tut.mode().iloc[0])
y_pred_maj = np.full_like(y_test_tut, fill_value=maj_class)
maj_test_acc = float(accuracy_score(y_test_tut, y_pred_maj))

# Tutorial XGBClassifier
tut_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    eval_metric='logloss',
    random_state=SEED,
    n_jobs=-1,
)
tut_model.fit(X_train_tut, y_train_tut)

y_pred_tut = tut_model.predict(X_test_tut)
y_proba_tut = tut_model.predict_proba(X_test_tut)[:, 1]
y_train_pred_tut = tut_model.predict(X_train_tut)
y_train_proba_tut = tut_model.predict_proba(X_train_tut)[:, 1]

tut_test_acc = float(accuracy_score(y_test_tut, y_pred_tut))
tut_test_auc = float(roc_auc_score(y_test_tut, y_proba_tut))
tut_train_acc = float(accuracy_score(y_train_tut, y_train_pred_tut))
tut_train_auc = float(roc_auc_score(y_train_tut, y_train_proba_tut))
tut_prec = float(precision_score(y_test_tut, y_pred_tut))
tut_rec = float(recall_score(y_test_tut, y_pred_tut))
tut_f1 = float(f1_score(y_test_tut, y_pred_tut))
tut_cm = confusion_matrix(y_test_tut, y_pred_tut).tolist()

print(f'Majority baseline test accuracy (always predict {maj_class}): {maj_test_acc:.4f}')
print(f'Tutorial XGBoost Test Accuracy : {tut_test_acc:.4f} (expected ~0.6357)')
print(f'Tutorial XGBoost Test ROC-AUC  : {tut_test_auc:.4f} (expected ~0.694)')
print(f'Tutorial XGBoost Train Accuracy: {tut_train_acc:.4f} (expected ~0.735)')
print(f'Tutorial XGBoost Train ROC-AUC : {tut_train_auc:.4f}')
print(f'Tutorial Test Precision / Recall / F1: {tut_prec:.4f} / {tut_rec:.4f} / {tut_f1:.4f}')
print(f'Tutorial Confusion Matrix [[TN, FP], [FN, TP]]: {tut_cm}')

assert abs(tut_test_acc - 0.6357) <= 0.01, f'Test acc {tut_test_acc} outside calibration tolerance'
assert abs(tut_test_auc - 0.694) <= 0.01, f'Test AUC {tut_test_auc} outside calibration tolerance'
assert abs(tut_train_acc - 0.735) <= 0.01, f'Train acc {tut_train_acc} outside calibration tolerance'

RESULTS['tutorial_baseline'] = {
    'majority_class': maj_class,
    'majority_test_acc': round(maj_test_acc, 6),
    'test_acc': round(tut_test_acc, 6),
    'test_auc': round(tut_test_auc, 6),
    'train_acc': round(tut_train_acc, 6),
    'train_auc': round(tut_train_auc, 6),
    'test_precision': round(tut_prec, 6),
    'test_recall': round(tut_rec, 6),
    'test_f1': round(tut_f1, 6),
    'confusion_matrix': tut_cm,
}
save_results()

/tmp/ipykernel_10663/3246477047.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols_tut = df_tut.select_dtypes(include='object').columns.tolist()


Majority baseline test accuracy (always predict 0): 0.5044
Tutorial XGBoost Test Accuracy : 0.6357 (expected ~0.6357)
Tutorial XGBoost Test ROC-AUC  : 0.6938 (expected ~0.694)
Tutorial XGBoost Train Accuracy: 0.7349 (expected ~0.735)
Tutorial XGBoost Train ROC-AUC : 0.8151
Tutorial Test Precision / Recall / F1: 0.6288 / 0.6468 / 0.6377
Tutorial Confusion Matrix [[TN, FP], [FN, TP]]: [[9454, 5677], [5252, 9617]]


> **So what:** Our pipeline reproduces the tutorial benchmark (`0.6357` test accuracy, `0.6940` test AUC vs. `0.5044` majority baseline, and `0.7347` train accuracy showing ~10 percentage points of overfitting at `max_depth=6`). Full 5-fold CV model comparisons (`M0`–`M6`) follow in Phase 6.

---
## 8. Model Evaluation & Interpretation
*Pending Phase 7 (T7.1–T7.8) — Confusion matrix, top-k targeting table & lift curve, calibration, gain/permutation importance, partial dependence, ablation, and lever table.*

---
## 9. Business-Impact Model & Realism Adjustment
*Pending Phase 8 (T8.1–T8.8) — Label-shift re-basing (49.56% to 5%/10%), net impact scenarios, optimal k*, break-even s*, tornado sensitivity, and Lane 2 pilot sizing.*

---
## 10. Export Summary & Verification
*Pending Phase 9 (T9.3) — Final verification of outputs/results.json, figures, and tables.*

---
## 11. Conclusion, Limitations & Next Steps
*Pending Phase 9 — Key takeaways, analytical limitations, ethical safeguards, and 90-day pilot roadmap.*